In [37]:
import pandas as pd
# import numpy as np
import plotly.express as px
# import plotly.graph_objects as go
from plotly.subplots import make_subplots
# from scipy.stats import gaussian_kde

In [38]:
df = pd.read_csv("../benchmarks1/output/performance_detailed.csv")

def parse_reduced_size(s):
    try:
        r, c = map(int, s.split('×'))
        return pd.Series({'rows': r, 'cols': c, 'size': r * c})
    except:
        return pd.Series({'rows': 0, 'cols': 0, 'size': 0})

df = pd.concat([df, df['reduced_size'].apply(parse_reduced_size)], axis=1)
df = df.drop(columns=['reduced_size']+['file'])

df_failed = df.copy()
indices_to_drop = df_failed[df_failed['timeout'] != True].index
df_failed = df_failed.drop(indices_to_drop)

indices_to_drop = df[df['timeout']].index
df = df.drop(indices_to_drop)

to_drop_cols = [col for col in df.columns if not df[col].any() and df[col].dtype == 'bool'] # all columns that are empty and boolean
df = df.drop(columns=['matrix_size'] + to_drop_cols)
df_failed = df_failed.drop(columns=['matrix_size'] + to_drop_cols)

fig = px.parallel_coordinates(df, color="levels_explored", labels={"hypotheses_generated": "Ipotesi",
                  "levels_explored": "Livelli esplorati", "max_level_size": "Massima ampiezza",
                  "max_level_size": "Dimensione file", "ones_count": "numeri di uno", },
                    color_continuous_midpoint=4)
fig.show()

# fig = px.parallel_categories(df, color="mhs_count", color_continuous_scale=px.colors.sequential.Inferno)
# fig.show()

df

,mhs_count,computation_time,hypotheses_generated,levels_explored,max_level_size,file_size_mb,ones_count,rows,cols,size
0,271,1.695186,2867,3,2356,0.001053,54,3,33,99
1,95,0.014971,260,2,236,0.000925,29,2,24,48
2,23,0.011114,187,2,163,0.000925,34,2,24,48
3,1,0.002052,55,2,45,0.000924,11,2,10,20
4,14,0.001706,59,2,44,0.000925,25,2,15,30
...,...,...,...,...,...,...,...,...,...,...
168,1643,44.796836,3731,2,3645,0.007253,91,2,86,172
260,30,0.028851,103,2,88,0.011719,20,2,15,30
261,55,0.030231,188,2,168,0.011719,25,2,20,40
262,50,0.045471,221,2,198,0.011717,28,2,23,46


In [39]:
df_failed

,mhs_count,computation_time,hypotheses_generated,levels_explored,max_level_size,file_size_mb,ones_count,rows,cols,size
10,0,61.027075,5895,3,5899,0.001571,104,7,34,238
11,0,62.726310,4581,3,14836,0.001443,105,6,46,276
12,0,61.679561,5330,3,6545,0.001698,88,8,35,280
13,0,60.293351,4680,3,9130,0.001443,74,6,39,234
15,0,61.689120,5228,3,14896,0.001827,157,9,47,423
...,...,...,...,...,...,...,...,...,...,...
345,0,101.951036,200,1,544,0.132941,2650,74,544,40256
346,0,96.306368,200,1,533,0.116122,1832,64,533,34112
347,0,64.458255,200,1,455,0.146399,1814,82,455,37310
348,0,136.652365,200,1,617,0.193519,3778,110,617,67870


In [40]:
def clean_dataset(df):
    df_numeric = df.copy()
    for col in df_numeric.columns:
        original_dtype = df_numeric[col].dtype
        df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce')
        if df_numeric[col].isnull().all() and not pd.api.types.is_numeric_dtype(original_dtype):
            print(f"Avviso: La colonna '{col}' è diventata tutta NaN dopo la conversione. Probabilmente non era numerica.")

    initial_cols = set(df_numeric.columns)
    df_numeric.dropna(axis=1, how='all', inplace=True)
    dropped_cols_all_nan = list(initial_cols - set(df_numeric.columns))
    if dropped_cols_all_nan:
        print(f"\nColonne rimosse perché interamente NaN dopo conversione: {dropped_cols_all_nan}")
    else:
        print("\nNessuna colonna rimossa perché interamente NaN dopo conversione.")
    return df_numeric

def extract_stats(col, plot_data):
    mean_val = plot_data[col].mean()
    std_val = plot_data[col].std()
    median_val = plot_data[col].median()
    min_val = plot_data[col].min()
    max_val = plot_data[col].max()

    stats_text = (
        f'µ={mean_val:.2f}<br>'
        f'σ={std_val:.2f}<br>'
        f'median={median_val:.2f}<br>'
        f'min={min_val:.2f}<br>'
        f'max={max_val:.2f}'
    )
    
    return stats_text

def add_annotation(stats_text, fig):
    fig.add_annotation(
        xref="paper",
        yref="paper",
        x=1.05,
        y=0.5,
        text=stats_text,
        showarrow=False,
        font=dict(
            size=12,
            color="black"
        ),
        align="left",
        xanchor='left',
        yanchor='middle',
        bgcolor="lightgrey",
        bordercolor="black",
        borderwidth=1,
        borderpad=5
        )


In [41]:
df_numeric = clean_dataset(df)

df_numeric.info()
df_numeric.head()

cols_to_plot = [col for col in df_numeric.columns if pd.api.types.is_numeric_dtype(df_numeric[col])]

if not cols_to_plot:
    print("\nErrore: Nessuna colonna numerica valida trovata per il plotting.")
    exit()

for col in cols_to_plot:
    plot_data = df_numeric[[col]].dropna()

    try:
        stats_text = extract_stats(col, plot_data)

        fig = px.histogram(
            plot_data,
            x=col,
            histnorm='density',
            color_discrete_sequence=['red'],
            opacity=0.75,
            title=f'{col}',
            labels={col: col.replace("_", " ").title(), 'density': 'Density'},
            width=900, height=500,
            nbins=20, 
            marginal="violin"
        )
        
        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')

        add_annotation(stats_text, fig)

        fig.update_layout(
            margin=dict(
                l=80,  
                r=350, 
                b=80, 
                t=80   
            ),
        )

        fig.show()

    except Exception as e:
        print(f"  ERRORE CRITICO: Non è stato possibile generare il grafico per '{col}'. Errore: {e}")
        print(f"  Dati problematici per '{col}':")
        print(plot_data[col].describe())
        print(plot_data[col].head())
        print(f"  Conteggio NaN: {plot_data[col].isnull().sum()}")


Nessuna colonna rimossa perché interamente NaN dopo conversione.
<class 'pandas.core.frame.DataFrame'>
Index: 98 entries, 0 to 265
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   mhs_count             98 non-null     int64  
 1   computation_time      98 non-null     float64
 2   hypotheses_generated  98 non-null     int64  
 3   levels_explored       98 non-null     int64  
 4   max_level_size        98 non-null     int64  
 5   file_size_mb          98 non-null     float64
 6   ones_count            98 non-null     int64  
 7   rows                  98 non-null     int64  
 8   cols                  98 non-null     int64  
 9   size                  98 non-null     int64  
dtypes: float64(2), int64(8)
memory usage: 8.4 KB


In [42]:
df_failed_numeric = clean_dataset(df_failed)
df_failed_numeric = df_failed_numeric.drop(columns=['mhs_count'])
cols_to_plot = [col for col in df_failed_numeric.columns if pd.api.types.is_numeric_dtype(df_failed_numeric[col])]

if not cols_to_plot:
    print("\nErrore: Nessuna colonna numerica valida trovata per il plotting.")
    exit()

for col in cols_to_plot:
    plot_data = df_failed_numeric[[col]].dropna()

    try:
        stats_text = extract_stats(col, plot_data)

        fig = px.histogram(
            plot_data,
            x=col,
            histnorm='density',
            color_discrete_sequence=['blue'],
            opacity=0.75,
            title=f'{col}',
            labels={col: col.replace("_", " ").title(), 'density': 'Density'},
            width=900, height=500,
            nbins=20, 
            marginal="violin"
        )
        

        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')

        add_annotation(stats_text, fig)
        
        fig.update_layout(
            margin=dict(
                l=80,  
                r=350, 
                b=80, 
                t=80   
            ),
        )

        fig.show()

    except Exception as e:
        print(f"  ERRORE CRITICO: Non è stato possibile generare il grafico per '{col}'. Errore: {e}")
        print(f"  Dati problematici per '{col}':")
        print(plot_data[col].describe())
        print(plot_data[col].head())
        print(f"  Conteggio NaN: {plot_data[col].isnull().sum()}")


Nessuna colonna rimossa perché interamente NaN dopo conversione.


In [43]:
correlation_matrix = df.corr()

fig = px.imshow(correlation_matrix,
                text_auto=True, 
                # color_continuous_scale=px.colors.sequential.RdBu,
                # range_color=[-1, 1],
                title='Matrice di Correlazione tra Variabili',
                labels=dict(x="Variabile 1", y="Variabile 2", color="Correlazione"),
                width=1600, 
                height=1600 
               )

fig.update_xaxes(side="top") 
fig.update_layout(
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    xaxis_zeroline=False,
    yaxis_zeroline=False,
    coloraxis_colorbar=dict(
        title="Correlazione", 
        tickvals=[-1, 0, 1], 
        ticktext=["-1 (Negativa Forte)", "0 (Nessuna)", "1 (Positiva Forte)"]
    )
)

fig.show()


I valori che hanno un maggior indice di correlazione sono

*   `rows` e `level_explored` -> indice pari a 1 (grado massimo di correlazione)
*   `max_level_size` e `computational_time` -> indice pari a 0.87
*   `cols` e `mhs_count` -> indice pari a 0.85

Infatti si può intuitivamente comprendere che maggiore è il tempo coputazionale dedicato alla ricerca e risulta più probabile riuscire ad espandere la dimensione dell'ultimo livello esplorato.

Protrebbe quindi aver senso vedere quale sai l'andamento fra le due features correlate attraverso uno scatter.

In [49]:
fig = px.scatter(
    df.sort_values(by='rows'),                 
    x='rows',                      
    y='levels_explored',                  
    color='levels_explored',
    size='levels_explored',                 
    hover_data=['rows', 'levels_explored'],
    title='Scatter Plot Interattivo: levels_explored vs rows',
    labels={'rows': 'Righe della matrice', 'levels_explored': 'Livelli esplorati'},
    marginal_x="box"
)

fig.show()

In [44]:
df['mhs_cat'] = pd.cut(df['mhs_count'], bins=5, labels=['1','2','3','4','5'], right=True, include_lowest=True)
fig = px.scatter_matrix(df, height=1600, width=1600, color="mhs_cat")
fig.show()

In [ ]:
fig = px.scatter(
    df,                 
    x='size',                      
    y='computation_time',                  
    color='mhs_count',
    size='mhs_count',                 
    hover_data=['size', 'computation_time', 'mhs_count'],
    title='Scatter Plot Interattivo: computation_time vs size',
    labels={'size': 'Elementi totali della matrice', 'computation_time': 'Tempo computazionale (s)', 'mhs_count': 'Numero di MHS'}
    # marginal_x="box"
)

fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='size',
    y='computation_time',
    color='ones_count',  # Colora i punti in base al numero di '1'
    size='ones_count', # La dimensione del punto indica il numero di '1' nella matrice
    hover_data=['mhs_count', 'hypotheses_generated', 'levels_explored', 'file_size_mb', 'ones_count', 'cols'],
    title='Tempo di Calcolo vs. Dimensione della Matrice (Colorato per 1)',
    labels={'size': 'Dimensione Matrice (righe*colonne)', 'computation_time': 'Tempo di Calcolo (s)'},
    log_y=True # Utile se il tempo di calcolo cresce esponenzialmente
)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='hypotheses_generated',
    y='computation_time',
    color='size', # Colora in base alla dimensione del problema
    size='mhs_count', # La dimensione del punto indica il numero di MHS trovati
    hover_data=['mhs_count', 'levels_explored', 'max_level_size', 'rows', 'cols', 'file_size_mb'],
    title='Tempo di Calcolo vs. Ipotesi Generate (Colorato per Dimensione Matrice)',
    labels={'hypotheses_generated': 'Ipotesi Generate', 'computation_time': 'Tempo di Calcolo (s)'},
    log_x=True, # Utile se le ipotesi generate crescono molto
    log_y=True
)
fig.show()

In [ ]:
fig = px.box(
    df,
    x='rows',
    y='computation_time',
    color='cols', # Mostra box plot separati per 'rows', colorati per 'cols'
    points="all", # Mostra tutti i punti dati oltre al box plot
    hover_data=['mhs_count', 'size', 'file_size_mb', 'ones_count'],
    title='Distribuzione del Tempo di Calcolo per Numero di Righe e Colonne',
    labels={'rows': 'Numero di Righe', 'computation_time': 'Tempo di Calcolo (s)'},
    log_y=True
)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x='size',
    y='computation_time',
    color='ones_count', # Colora in base al numero di '1'
    facet_col='cols', # Crea colonne separate per ogni valore di 'cols'
    facet_row='rows', # Crea righe separate per ogni valore di 'rows'
    hover_data=['mhs_count', 'hypotheses_generated', 'levels_explored', 'file_size_mb'],
    title='Tempo di Calcolo vs. Dimensione Matrice per Righe e Colonne',
    labels={'size': 'Dimensione Matrice', 'computation_time': 'Tempo di Calcolo (s)'},
    log_y=True,
    height=800 # Aumenta l'altezza per una migliore visualizzazione dei facet
)
fig.show()

In [ ]:
df_sorted = df.sort_values(by=['rows', 'cols', 'size'])

fig = px.line(
    df_sorted,
    x='size',                  # Variabile sull'asse X (dimensione della matrice)
    y='computation_time',      # Variabile sull'asse Y (tempo di calcolo)
    color='rows',              # Crea una linea separata e colorata per ogni valore di 'rows'
    line_dash='cols',          # (Opzionale) Aggiunge uno stile di linea diverso per ogni valore di 'cols'
    hover_data=['mhs_count', 'hypotheses_generated', 'levels_explored', 'file_size_mb', 'ones_count', 'rows', 'cols'],
    title='Tempo di Calcolo vs. Dimensione Matrice per Diverse Config. (rows/cols)',
    labels={'size': 'Dimensione Matrice (righe*colonne)', 'computation_time': 'Tempo di Calcolo (s)',
            'rows': 'Num. Righe', 'cols': 'Num. Colonne'},
    log_y=True,                # Utile se il tempo di calcolo cresce esponenzialmente
    markers=True               # Mostra un marcatore per ogni punto dato
)

fig.update_traces(mode='lines+markers') # Assicurati che vengano mostrati sia linee che marcatori

fig.show()

In [ ]:
fig = px.line(
    df.sort_values('cols'),
    x='cols',
    y='computation_time',
    line_dash='rows',
    title='Computation Time vs. Number of Columns',
    labels={'cols': 'Number of Columns', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [ ]:
fig = px.line(
    df.sort_values('ones_count'),
    x='ones_count',
    y='computation_time',
    title='Computation Time vs. Number of Ones',
    labels={'ones_count': 'Number of Ones', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [ ]:
fig = px.line(
    df.sort_values('cols'),
    x='cols',
    y='computation_time',
    title='Computation Time vs. Number of Columns',
    labels={'cols': 'Number of Columns', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

# df.sort_values('cols').plot(x='cols', y='computation_time', marker='o', legend=False)
# plt.xlabel('Numero colonne')
# plt.ylabel('Tempo di calcolo (s)')
# plt.title('Tempo vs Numero colonne')
# plt.grid(True)
# plt.show()

In [ ]:
fig = px.line(
    df.sort_values('rows'),
    x='rows',
    y='computation_time',
    title='Computation Time vs. Number of Rows',
    labels={'rows': 'Number of Rows', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

# df.sort_values('rows').plot(x='rows', y='computation_time', marker='o', legend=False)
# plt.xlabel('Numero righe')
# plt.ylabel('Tempo di calcolo (s)')
# plt.title('Tempo vs Numero righe')
# plt.grid(True)
# plt.show()

In [ ]:
fig = px.line(
    df.sort_values('size'),
    x='size',
    y='computation_time',
    title='Computation Time vs. Number of Elements',
    labels={'size': 'Number of Elements', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [ ]:
fig = px.line(
    df.sort_values('mhs_count'),
    x='mhs_count',
    y='computation_time',
    title='Computation Time vs. Number of MHS found',
    labels={'mhs_count': 'Number of MHS found', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [ ]:
fig = px.line(
    df.sort_values('mhs_count'),
    x='mhs_count',
    y='size',
    title='Size vs. Number of MHS found',
    labels={'mhs_count': 'Number of MHS found', 'size': 'Size'},
    markers=True
)

fig.show()

In [ ]:
fig = px.line(
    df.sort_values('file_size_mb'),
    x='file_size_mb',
    y='computation_time',
    title='Computation Time vs. File Size',
    labels={'file_size_mb': 'File Size (MB)', 'computation_time': 'Computation Time (s)'},
    markers=True,
    height=800,  # Altezza del grafico
    width=1600   # Larghezza del grafico
)

fig.show()

In [ ]:
fig = px.line(
    df.sort_values('levels_explored'),
    x='levels_explored',
    y='computation_time',
    title='Computation Time vs. Levels Explored',
    labels={'levels_explored': 'Levels Explored', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [ ]:
fig = px.line(
    df.sort_values('max_level_size'),
    x='max_level_size',
    y='computation_time',
    title='Computation Time vs. Max Level Size',
    labels={'max_level_size': 'Max Level Size', 'computation_time': 'Computation Time (s)'},
    markers=True,
    height=800,
    width=1600  
)

fig.show()